# Dilithium

Dilithium es una firma digital que no puede ser rota por ordenadores cuánticos (a diferencia de RSA o ECDSA).

Funciona así:
- Tenemos una **matriz pública A** y un **secreto s** (vector con coeficientes muy pequeños)
- Publicamos **t = A·s** (fácil de calcular, imposible de invertir)
- Para firmar: generamos un valor aleatorio **y**, calculamos **z = y + c·s** donde c viene de hashear el mensaje
- Para verificar: comprobamos que **A·z - c·t ≈ A·y** (solo funciona si z se construyó con el secreto correcto)

La seguridad se basa en que **encontrar s a partir de A y t es imposible** (problema de retículas, resistente a Shor).


## Implementación

Usamos SageMath. Todo opera en el anillo de polinomios $R_q = \mathbb{Z}_q[x]/(x^{256}+1)$ con q = 8.380.417.

Primero cargamos los parámetros y funciones auxiliares:

In [1]:
import hashlib, struct, os, time

# Parámetros
q = 8380417; n = 256; k = l = 4
eta = 2; gamma1 = 2^17; gamma2 = (q-1)//88; beta_bound = 78; tau = 39

# Anillo de polinomios
R.<x> = PolynomialRing(Integers(q))
Rq = R.quotient(x^n + 1, 'a')

# Función para hash
def H(data, nb=32): return hashlib.shake_256(data).digest(nb)

# Muestreo de polinomios con coeficientes pequeños
def sample_small(seed, nonce):
    stream = H(seed+struct.pack('<H',nonce), 4*n)
    c=[]; p=0
    while len(c)<n:
        b=stream[p]; p+=1
        for nib in [b&0xF,b>>4]:
            if nib<=4 and len(c)<n: c.append(eta-nib)
    return Rq(c)

# Muestreo de polinomios con coeficientes en un rango más amplio (construcción de la Y)
def sample_mask(seed, nonce):
    stream = H(seed+struct.pack('<H',nonce), 592)
    c=[]
    for i in range(64):
        blk=int.from_bytes(stream[9*i:9*i+9],'little')
        for j in range(4):
            v=(blk>>(18*j))&0x3FFFF
            c.append(Integer(gamma1-1-(v%(2*gamma1-1))))
    return Rq(c[:n])

# Función para construir la matriz A 
def expand_A(rho):
    A=[]
    for i in range(k):
        row=[]
        for j in range(l):
            s=hashlib.shake_128(rho+struct.pack('<BB',j,i)).digest(3*n+300)
            c=[]; p=0
            while len(c)<n:
                v=s[p]+(s[p+1]<<8)+((s[p+2]&0x7F)<<16); p+=3
                if v<q: c.append(v)
            row.append(Rq(c))
        A.append(row)
    return A



# Función para centrar un entero en el intervalo [-alpha//2, alpha//2]
def centrar(r, alpha):
    r0=Integer(r)%alpha
    return r0-alpha if r0>alpha//2 else r0


# Funciones para obtener los bits altos de un polinomio (similar a coger parte entera de un número real)
def poly_highbits(f, alpha):
    c=list(f.lift())+[0]*(n-len(list(f.lift())))
    res=[]
    for r in c:
        r=Integer(r)%q; r0=centrar(r,alpha)
        res.append(0 if r-r0==q-1 else (r-r0)//alpha)
    return res

def poly_lowbits(f, alpha):
    c=list(f.lift())+[0]*(n-len(list(f.lift())))
    return [centrar(Integer(r)%q, alpha) for r in c]

# Función para calcular la norma infinita de un polinomio
def norma_inf_vec(v):
    m=0
    for f in v:
        for c in list(f.lift()):
            m=max(m, abs(centrar(c,q)))
    return m

# Función para muestrear el desafío 
def sample_challenge(seed):
    stream=H(seed,8+tau*4); signos=int.from_bytes(stream[:8],'little')
    c=[0]*n; pos=8
    for i in range(n-tau,n):
        while True:
            if pos>=len(stream): stream+=H(seed+stream[-32:],256)
            j=stream[pos]; pos+=1
            if j<=i: break
        c[i]=c[j]; c[j]=q-1 if (signos>>(i-(n-tau)))&1 else 1
    return Rq(c)

def mat_vec(A,v):
    return [sum(A[i][j]*v[j] for j in range(l)) for i in range(k)]

def vec_add(a,b): 
    return [a[i]+b[i] for i in range(len(a))]

def vec_sub(a,b): 
    return [a[i]-b[i] for i in range(len(a))]

def scalar_vec(c,v): 
    return [c*vi for vi in v]
    
print("Listo ✓")

Listo ✓



## KeyGen

Genera las claves.

- Crea una matriz aleatoria **A**
- Elige secretos **s1, s2** con coeficientes pequeños (en [-2, 2])
- Calcula **t = A·s1 + s2**
- Publica **(A, t)**, guarda en secreto **(s1, s2)**

Seguridad: recuperar s1 de A y t es imposible (problema Module-LWE).

In [2]:
def keygen(seed=None):
    if seed is None: seed = os.urandom(32)
    exp = H(seed, 96)
    rho, rho_p, K = exp[:32], exp[32:64], exp[64:96]
    
    A  = expand_A(rho) # Matriz pública
    s1 = [sample_small(rho_p, i) for i in range(l)] # Secreto 1
    s2 = [sample_small(rho_p, l+i) for i in range(k)] # Secreto 2
    t  = vec_add(mat_vec(A, s1), s2) # t = A·s1 + s2

    return {'rho':rho, 't':t}, {'rho':rho, 'K':K, 's1':s1, 's2':s2}


## Sign

**Firmar un mensaje** demostrando que conoces s1 sin revelarlo.

1. Elige **y** aleatorio (grande, para ocultar s1)
2. Calcula **w = A·y**, extrae **w₁ = HighBits(w)**
3. Genera desafío **c = Hash(mensaje, w₁)**
4. Calcula respuesta **z = y + c·s1**
5. Si z es demasiado grande → **descarta y repite** (para no filtrar info de s1)

Firma = **(z, c)**

In [3]:
def sign(sk, mensaje):
    rho, K, s1, s2 = sk['rho'], sk['K'], sk['s1'], sk['s2']
    A = expand_A(rho)
    if isinstance(mensaje, str): mensaje = mensaje.encode()
    mu = H(rho + mensaje, 64)
    seed_y = H(K + mu, 64)
    nonce = intentos = 0
    
    while True:
        intentos += 1
        y = [sample_mask(seed_y, nonce+i) for i in range(l)]; nonce += l

        w = mat_vec(A, y) # Compromiso
        w1 = [poly_highbits(wi, 2*gamma2) for wi in w] # Parte alta
        w1b = b''.join(struct.pack('<i',c) for r in w1 for c in r)
        c_seed = H(mu + w1b, 32)
        c = sample_challenge(c_seed) # Desafío
        
        z = vec_add(y, scalar_vec(c, s1)) # Respuesta

        # Si z es grande, podría revelar s1 -> FIAT SHAMIR
        if norma_inf_vec(z) >= gamma1 - beta_bound: 
            continue
        
        w_cs2 = vec_sub(w, scalar_vec(c, s2))

        if max(max(abs(v) for v in poly_lowbits(p,2*gamma2)) for p in w_cs2) >= gamma2-beta_bound: 
            continue
        
        print(f"  Firma OK ({intentos} intento/s)")
        return {'z':z, 'c_seed':c_seed, 'c':c}


## Verificar

**Verifica la firma** usando solo la clave pública.

1. Comprueba que **z no sea grande**
2. Calcula **w' = A·z - c·t**
3. Recalcula el hash y comprueba que **coincide con c**

Funciona porque: A·z - c·t = A·(y+c·s1) - c·(A·s1+s2) = **A·y - c·s2 ≈ A·y** (c·s2 es pequeño → HighBits no cambia).

In [4]:
def verify(pk, mensaje, firma):
    rho, t = pk['rho'], pk['t']
    z, c_seed, c = firma['z'], firma['c_seed'], firma['c']
    
    if norma_inf_vec(z) >= gamma1 - beta_bound:
        print("  ✗ INVÁLIDA"); return False
    
    A = expand_A(rho)
    w_prima = vec_sub(mat_vec(A, z), scalar_vec(c, t))             # w' = Az - ct
    w1_prima = [poly_highbits(wi, 2*gamma2) for wi in w_prima]
    
    if isinstance(mensaje, str): mensaje = mensaje.encode()
    mu = H(rho + mensaje, 64)
    w1b = b''.join(struct.pack('<i',c) for r in w1_prima for c in r)
    
    if c_seed == H(mu + w1b, 32):
        print("  ✓ VÁLIDA")
        return True
    else:
        print("  ✗ INVÁLIDA")
        return False


## Prueba

In [5]:
# Generar claves
pk, sk = keygen(seed=b"semilla_dilithium_jorge!")
print("Claves generadas")

# Firmar
msg = "Mensaje original firmado con Dilithium"
firma = sign(sk, msg)

# Verificar (original)
print(f"\nVerificar '{msg}':")
verify(pk, msg, firma)

# Verificar (alterado)
print(f"\nVerificar 'Mensaje MODIFICADO':")
verify(pk, "Mensaje MODIFICADO", 
firma)

Claves generadas
  Firma OK (2 intento/s)

Verificar 'Mensaje original firmado con Dilithium':
  ✓ VÁLIDA

Verificar 'Mensaje MODIFICADO':
  ✗ INVÁLIDA


False